# Prediction API — Testes

Testa todos os endpoints de `/prediction`:
- `POST /prediction/predict` — Predição (leitura de predições persistidas)
- `GET /prediction/backtest/{veiculo_id}?target=` — Backtest

> A predição com persistência é executada automaticamente durante a ingestão (`/profiling/ingest`).

In [13]:
import httpx
import time

BASE_URL = "http://localhost:8010"
client = httpx.Client(base_url=BASE_URL, timeout=300)

## 1. Predição para veículos específicos (sem persistência)

In [97]:
TARGET = "km"
VEHICLE_IDS = [4, 11, 38]

resp = client.post("/prediction/predict", json={"target": TARGET, "vehicle_ids": VEHICLE_IDS})
print(f"Status: {resp.status_code}")
data = resp.json()
data.keys()

Status: 200


dict_keys(['target', 'predictions_daily', 'predictions_heads', 'type_probabilities', 'not_found'])

In [98]:
# Inspecionar predições daily
import pandas as pd

if data.get("predictions_daily"):
    df_daily = pd.DataFrame(data["predictions_daily"])
    display(df_daily)
else:
    print("Sem predições daily")

,veiculo_id,data,prediction
0,4,2025-09-15,18.480859
1,4,2025-09-16,18.928609
2,4,2025-09-17,19.600160
3,4,2025-09-18,18.699611
4,4,2025-09-19,17.258997
5,4,2025-09-20,0.094911
6,4,2025-09-21,0.015563
7,11,2025-09-15,21.136847
8,11,2025-09-16,29.929132
9,11,2025-09-17,32.393983


In [99]:
# Inspecionar predições heads
if data.get("predictions_heads"):
    df_heads = pd.DataFrame(data["predictions_heads"])
    display(df_heads)
else:
    print("Sem predições heads")

,veiculo_id,dt_inicio,dt_fim,prediction
0,4,2025-09-15,2025-09-21,122.575091
1,4,2025-09-22,2025-09-28,121.133051
2,4,2025-09-29,2025-10-05,118.716961
3,4,2025-10-06,2025-10-12,113.378158
4,11,2025-09-15,2025-09-21,223.664936
5,11,2025-09-22,2025-09-28,227.034947
6,11,2025-09-29,2025-10-05,231.372337
7,11,2025-10-06,2025-10-12,225.287304
8,38,2025-09-15,2025-09-21,7.755767
9,38,2025-09-22,2025-09-28,11.448538


In [77]:
# Veículos não encontrados
if data.get("not_found"):
    pd.DataFrame(data["not_found"])
else:
    print("Todos os veículos encontrados")

Todos os veículos encontrados


In [ ]:
# Qualidade dos veículos
if data.get("vehicle_quality"):
    df_quality = pd.DataFrame(data["vehicle_quality"])
    display(df_quality)
else:
    print("Sem informação de qualidade")

## 2. Predição para todos os veículos

In [ ]:
resp = client.post("/prediction/predict", json={"target": TARGET})
print(f"Status: {resp.status_code}")
data_all = resp.json()

print(f"Daily: {len(data_all.get('predictions_daily', []))} registos")
print(f"Heads: {len(data_all.get('predictions_heads', []))} registos")
print(f"Type probs: {len(data_all.get('type_probabilities', []))} veículos")
print(f"Not found: {len(data_all.get('not_found', []))} veículos")

## 3. Backtest

Compara predições persistidas com valores reais de `daily_activity`.

**Requer** que a ingestão (`/profiling/ingest`) já tenha sido executada.

In [100]:
VEICULO_ID = 38

resp = client.get(f"/prediction/backtest/{VEICULO_ID}", params={"target": TARGET})
print(f"Status: {resp.status_code}")
bt = resp.json()
bt.keys()

Status: 200


dict_keys(['veiculo_id', 'target', 'daily', 'heads'])

In [101]:
# Daily: comparação data a data
if bt.get("daily"):
    df_bt_daily = pd.DataFrame(bt["daily"])
    df_bt_daily["error"] = df_bt_daily["actual"] - df_bt_daily["predicted"]
    display(df_bt_daily)
else:
    print("Sem dados daily")

,data,actual,predicted,error
0,2025-08-25,0.0,0.811836,-0.811836
1,2025-08-26,0.0,2.724117,-2.724117
2,2025-08-27,0.0,2.422089,-2.422089
3,2025-08-28,0.0,1.100030,-1.100030
4,2025-08-29,0.0,0.000000,0.000000
5,2025-08-30,0.0,0.000000,0.000000
6,2025-08-31,0.0,0.000000,0.000000
7,2025-09-01,0.0,0.504580,-0.504580
8,2025-09-02,0.0,2.466886,-2.466886
9,2025-09-03,0.0,2.229481,-2.229481


In [102]:
# Heads: comparação por horizonte
if bt.get("heads"):
    df_bt_heads = pd.DataFrame(bt["heads"])
    df_bt_heads["error"] = df_bt_heads["actual"] - df_bt_heads["predicted"]
    display(df_bt_heads)
else:
    print("Sem dados heads")

,dt_inicio,dt_fim,actual,predicted,error
0,2025-08-25,2025-08-31,0.0,13.785154,-13.785154
1,2025-09-01,2025-09-07,0.0,11.512366,-11.512366
2,2025-09-08,2025-09-14,0.0,6.402493,-6.402493


In [96]:
# Testar backtest com veículo sem predições → 404
resp = client.get("/prediction/backtest/999999", params={"target": TARGET})
print(f"Status: {resp.status_code}")
resp.json()

Status: 404


{'detail': "Sem predições persistidas para veículo 999999 (target='km')."}

## 4. Testar com target 'h'

In [32]:
resp = client.post("/prediction/predict", json={"target": "h", "vehicle_ids": [101]})
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'target': 'h',
 'predictions_daily': [],
 'predictions_heads': [],
 'type_probabilities': [],
 'not_found': [{'veiculo_id': 101,
   'reason': 'Veículo não existe ou não tem atividade registrada'}]}